# 1. Setup, Dependencies & Reproducibility
Install required packages, import core libraries, set random seeds for reproducible runs, and select compute device.

In [2]:
import os
import sys
import random
import copy
import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoTokenizer
from sklearn.metrics import classification_report, accuracy_score, f1_score
from sklearn.utils.class_weight import compute_class_weight
from tqdm.auto import tqdm

# Add project root to path for src imports
repo_root = os.path.abspath(os.path.join(os.path.dirname('__file__'), '../..'))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from src.utils import seed_everything, load_config
seed_everything(2025)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device configured: {device} | Seed: {seed_val}')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 99.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 615.4/615.4 kB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 52.2 MB/s eta 0:00:00
Device configured: cuda


# 2. Specialized VSFC Ontology Knowledge Graph Engine
Initializes `VSFCOntologyEngine` tailored for 3-class sentiment analysis (Negative, Neutral, Positive) via deep semantic inference and polarity injection.

In [3]:
# Initialize VSFCOntologyEngine using relative path to ontology RDF
from src.ontology_engine import OntologyEngine, VSFCOntologyEngine
from src.utils import load_config

config_path = os.path.join(repo_root, 'configs/vsfc.yaml')
config = load_config(config_path)
onto_cfg = config.get('ontology', {})

ontology_path = os.path.join(repo_root, onto_cfg.get('rdf_path', 'ontology/ekman_appraisal_ontology.rdf'))

ontengine = VSFCOntologyEngine(
    rdf_path=ontology_path,
    default_conf=onto_cfg.get('default_conf', 0.9),
    negation_attenuation=onto_cfg.get('negation_attenuation', 0.4),
    alpha_similarity=onto_cfg.get('alpha_similarity', 0.2),
    verbose=True,
)


Loading RDF knowledge graph from: /kaggle/input/datasets/minkduck/kg-ekman6/ekman6_4_9.rdf
Loading Knowledge Graph for VSFC: /kaggle/input/datasets/minkduck/kg-ekman6/ekman6_4_9.rdf
Loaded 91005 triples.


# 3. Vector Distribution Inspection
Inspects 24-dimensional continuous feature vectors across representative test sentences and validates activation patterns.

In [4]:
COLS_24D = (
    ["Anger", "Disgust", "Fear", "Happiness", "Neutral", "Sadness", "Surprise"]
    + [
        "Pleas",
        "Unpleas",
        "Loss",
        "Danger",
        "Unpred",
        "Sudden",
        "GoalObs",
        "Unfair",
        "Other",
    ]
    + ["Int_Hi", "Int_Me", "Int_Lo"]
    + ["Pos", "Neg", "Neu"]
    + ["NegHit", "NegCnt"]
)


def inspect_vectors(texts, engine):
    """Build a structured inspection DataFrame for ontology feature vectors."""
    data = []
    for txt in texts:
        vec = engine.getvector(txt)
        if isinstance(vec, tuple):
            vec = vec[0]
        vec_rounded = np.round(vec, 2)
        row = {"Sentence": txt}
        for i, val in enumerate(vec_rounded):
            row[COLS_24D[i]] = val
        data.append(row)
    return pd.DataFrame(data)


sample_texts = [
    "Món này ngon tuyệt vời",
    "Thái độ phục vụ quá tệ hại",
    "Không ngon lắm",
    "Món ăn bình thường, không có gì đặc sắc",
    "Tôi rất thất vọng về cách làm việc",
    "Sợ quá đi mất",
    "Tôi chẳng thấy buồn cười gì cả",
]

df_vector_preview = inspect_vectors(sample_texts, ontengine)
display(
    df_vector_preview[
        ["Sentence", "Happiness", "Sadness", "Pos", "Neg", "Neu", "NegHit"]
    ]
)

,Sentence,Happiness,Sadness,Pos,Neg,Neu,NegHit
0,Món này ngon tuyệt vời,0.60,0.00,0.94,0.00,0.00,0.0
1,Thái độ phục vụ quá tệ hại,0.00,0.00,0.00,0.00,0.00,0.0
2,Không ngon lắm,0.00,0.00,0.00,0.00,0.00,1.0
3,"Món ăn bình thường, không có gì đặc sắc",0.00,0.00,0.00,0.00,0.74,1.0
4,Tôi rất thất vọng về cách làm việc,0.42,0.85,0.81,1.00,0.00,0.0
5,Sợ quá đi mất,0.00,0.00,0.00,0.98,0.00,0.0
6,Tôi chẳng thấy buồn cười gì cả,0.00,0.00,0.00,0.00,0.00,0.0


# 4. Dataset Loading & Preprocessing (UIT-VSFC)
Reads UIT-VSFC sentences and sentiment labels (0: Negative, 1: Neutral, 2: Positive), constructs PyTorch DataLoaders, and computes balanced class weights.

In [5]:
tokenizer = AutoTokenizer.from_pretrained("uitnlp/visobert")

# Locate UIT-VSFC dataset directory
BASE_PATH = os.path.join(repo_root, 'data', 'vsfc')
print(f"Using dataset path: {BASE_PATH}")


def load_vsfc_folder(folder_name):
    """Load UIT-VSFC sentence and sentiment annotation files."""
    sent_path = f"{BASE_PATH}/{folder_name}/sents.txt"
    label_path = f"{BASE_PATH}/{folder_name}/sentiments.txt"
    with open(sent_path, "r", encoding="utf-8") as f:
        sentences = [line.strip() for line in f.readlines()]
    with open(label_path, "r", encoding="utf-8") as f:
        labels = [int(line.strip()) for line in f.readlines()]
    min_len = min(len(sentences), len(labels))
    return pd.DataFrame(
        {"Sentence": sentences[:min_len], "Sentiment": labels[:min_len]}
    )


df_train = load_vsfc_folder("train")
df_valid = load_vsfc_folder("dev")
df_test = load_vsfc_folder("test")

print(
    f"Dataset loaded: Train={len(df_train)}, Dev={len(df_valid)}, Test={len(df_test)}"
)


class VSFC_OntologyDataset(Dataset):

    def __init__(self, df, tokenizer, engine, max_len=128):
        self.df = df
        self.tokenizer = tokenizer
        self.engine = engine
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        txt = str(self.df.iloc[idx]["Sentence"])
        lbl = int(self.df.iloc[idx]["Sentiment"])

        enc = self.tokenizer(
            txt,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )

        vec = self.engine.getvector(txt)
        if isinstance(vec, tuple):
            vec = vec[0]

        return {
            "input_ids": enc["input_ids"].flatten(),
            "attention_mask": enc["attention_mask"].flatten(),
            "ontology_features": torch.tensor(vec, dtype=torch.float),
            "labels": torch.tensor(lbl, dtype=torch.long),
        }


BATCH_SIZE = 32
train_loader = DataLoader(
    VSFC_OntologyDataset(df_train, tokenizer, ontengine),
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True,
)
valid_loader = DataLoader(
    VSFC_OntologyDataset(df_valid, tokenizer, ontengine), batch_size=BATCH_SIZE
)
test_loader = DataLoader(
    VSFC_OntologyDataset(df_test, tokenizer, ontengine), batch_size=BATCH_SIZE
)

# Compute class weights
train_labels = df_train["Sentiment"].tolist()
cls_w = compute_class_weight(
    "balanced", classes=np.unique(train_labels), y=train_labels
)
weights_tensor = torch.tensor(cls_w, dtype=torch.float).to(device)
print(f"Class Weights [Neg, Neu, Pos]: {weights_tensor.cpu().numpy()}")

config.json:   0%|          | 0.00/644 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/471k [00:00<?, ?B/s]

Using dataset path: /kaggle/input/datasets/minkduck/uit-vsfc/UIT-VSFC
Dataset loaded: Train=11426, Dev=1583, Test=3166
Class Weights [Neg, Neu, Pos]: [0.71524256 8.315866   0.6749365 ]


# 5. Neural-Symbolic Model Architectures
Defines neural-symbolic architectures for 3-class sentiment analysis:
1. `ViSoBERT_Fusion`: Standard ViSoBERT backbone supporting baseline, concatenation, and gated projection.
2. `ViSoBERT_DeepOntology_V2`: Multi-layer bottleneck projection with LayerNorm and Residual Gating.
3. `OntologyConceptProjector` & `ViSoBERT_Ontology_CrossAttention` (OCA): Token-level multi-head cross-attention over projected concept tokens.

In [6]:
# Import ViSoBERT fusion & cross-attention architectures from src.models
from src.models.visobert_fusion_models import (
    ViSoBERT_Fusion,
    ViSoBERT_DeepOntology,
    ViSoBERT_DeepOntology_V2,
    ViSoBERT_Residual_Fusion,
    OntologyConceptProjector,
    ViSoBERT_Ontology_CrossAttention,
)

ONT_INPUT_DIM = 24
print('✅ Models loaded from src.models.visobert_fusion_models')


# 6. Training & Evaluation Pipeline
Provides training utilities with validation monitoring, early stopping, and checkpoint restoration.

In [7]:
def evaluate(model, loader):
    """Evaluate accuracy and macro F1 score on a given DataLoader."""
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for b in loader:
            ids = b["input_ids"].to(device)
            m = b["attention_mask"].to(device)
            o = b["ontology_features"].to(device)
            l = b["labels"].to(device)

            out = model(ids, m, o)
            _, pr = torch.max(out, 1)
            preds.extend(pr.cpu().numpy())
            targets.extend(l.cpu().numpy())

    return accuracy_score(targets, preds), f1_score(
        targets, preds, average="macro"
    )


def train_experiment(model, name, epochs=15, patience=4):
    """Execute training loop with class-weighted cross-entropy and early stopping."""
    print(f"\nTraining Model: {name}")
    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)
    model.to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
    criterion = nn.CrossEntropyLoss(weight=weights_tensor)

    best_f1 = 0.0
    patience_counter = 0
    best_weights = copy.deepcopy(
        model.module.state_dict()
        if isinstance(model, nn.DataParallel)
        else model.state_dict()
    )

    for ep in range(epochs):
        model.train()
        loss_acc = 0.0
        for b in train_loader:
            optimizer.zero_grad()
            ids = b["input_ids"].to(device)
            m = b["attention_mask"].to(device)
            o = b["ontology_features"].to(device)
            l = b["labels"].to(device)

            loss = criterion(model(ids, m, o), l)
            loss.backward()
            optimizer.step()
            loss_acc += loss.item()

        acc, f1 = evaluate(model, valid_loader)
        avg_loss = loss_acc / len(train_loader)
        print(
            f"Epoch {ep + 1:02d} | Loss: {avg_loss:.4f} | Val F1: {f1:.4f} | Val Acc: {acc:.4f}",
            end=" ",
        )

        if f1 > best_f1:
            best_f1 = f1
            patience_counter = 0
            best_weights = copy.deepcopy(
                model.module.state_dict()
                if isinstance(model, nn.DataParallel)
                else model.state_dict()
            )
            print("(New Best)")
        else:
            patience_counter += 1
            print(f"(Patience: {patience_counter}/{patience})")

        if patience_counter >= patience:
            print(f"Early stopping triggered after {ep + 1} epochs.")
            break

    if isinstance(model, nn.DataParallel):
        model.module.load_state_dict(best_weights)
    else:
        model.load_state_dict(best_weights)

    return model

# 7. Model Training Across Configurations
Trains all comparative baseline and neural-symbolic models, including the proposed Ontology-Guided Cross-Attention architecture (OCA).

In [8]:
CLASS_NAMES = ["Negative", "Neutral", "Positive"]
NUM_CLASSES = len(CLASS_NAMES)

# 1. Baseline
m1 = ViSoBERT_Fusion(NUM_CLASSES, fusion_type="none")
m1 = train_experiment(m1, "1. Baseline (ViSoBERT)", epochs=15, patience=3)

# 2. Raw + Concat
m2 = ViSoBERT_Fusion(NUM_CLASSES, fusion_type="concat", ontology_dim=None)
m2 = train_experiment(m2, "2. Raw (24d) + Concat", epochs=15, patience=3)

# 3. Raw + Gate
m3 = ViSoBERT_Fusion(NUM_CLASSES, fusion_type="gate", ontology_dim=None)
m3 = train_experiment(m3, "3. Raw (24d) + Gate", epochs=15, patience=3)

# 4. Dense + Concat
m4 = ViSoBERT_Fusion(NUM_CLASSES, fusion_type="concat", ontology_dim=64)
m4 = train_experiment(m4, "4. Dense (64d) + Concat", epochs=15, patience=3)

# 5. Dense + Gate
m5 = ViSoBERT_Fusion(NUM_CLASSES, fusion_type="gate", ontology_dim=64)
m5 = train_experiment(m5, "5. Dense (64d) + Gate", epochs=15, patience=3)

# 6. Gated Residual Fusion (ViSoBERT_DeepOntology_V2)
m_res = ViSoBERT_DeepOntology_V2(
    n_classes=NUM_CLASSES,
    fusion_type="residual_gate",
    ont_input_dim=ONT_INPUT_DIM,
    hidden_dim=64,
)
m_res = train_experiment(
    m_res, "6. Gated Residual Fusion", epochs=15, patience=3
)

# 7. Proposed: Ontology-Guided Cross-Attention (OCA)
m_oca = ViSoBERT_Ontology_CrossAttention(
    model_name="uitnlp/visobert",
    n_classes=NUM_CLASSES,
    n_heads=8,
    dropout_p=0.3,
)
m_oca = train_experiment(
    m_oca,
    "7. ViSoBERT + Ontology Cross-Attention (OCA)",
    epochs=15,
    patience=3,
)

pytorch_model.bin:   0%|          | 0.00/390M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: uitnlp/visobert
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Training Model: 1. Baseline (ViSoBERT)


model.safetensors:   0%|          | 0.00/390M [00:00<?, ?B/s]

...

XLMRobertaModel LOAD REPORT from: uitnlp/visobert
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Training Model: 6. Gated Residual Fusion
Epoch 01 | Loss: 0.6392 | Val F1: 0.7303 | Val Acc: 0.8465 (New Best)
Epoch 02 | Loss: 0.3984 | Val F1: 0.7955 | Val Acc: 0.9084 (New Best)
Epoch 03 | Loss: 0.2733 | Val F1: 0.7661 | Val Acc: 0.8888 (Patience: 1/3)
Epoch 04 | Loss: 0.1858 | Val F1: 0.7999 | Val Acc: 0.9052 (New Best)
Epoch 05 | Loss: 0.1352 | Val F1: 0.7902 | Val Acc: 0.9065 (Patience: 1/3)
Epoch 06 | Loss: 0.0954 | Val F1: 0.8030 | Val Acc: 0.9135 (New Best)
Epoch 07 | Loss: 0.0661 | Val F1: 0.8001 | Val Acc: 0.9103 (Patience: 1/3)
Epoch 08 | Loss: 0.0693 | Val F1: 0.7906 | Val Acc: 0.9103 (Patience: 2/3)
Epoch 09 | Loss: 0.0581 | Val F1: 0.8090 | Val Acc: 0.9198 (New Best)
Epoch 10 | Loss: 0.0366 | Val F1: 0.7933 | Val Acc: 0.9210 (Patience: 1/3)
Epoch 11 | Loss: 0.0551 | Val F1: 0.7765 | Val Acc: 0.9122 (Patience: 2/3)
Epoch 12 | Loss: 0.0271 | Val F1: 0.7932 | Val Acc: 0.9191 (Patience: 3/3)
Early stopping triggered after 12 epochs.


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: uitnlp/visobert
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Training Model: 7. ViSoBERT + Ontology Cross-Attention (OCA)
Epoch 01 | Loss: 0.6595 | Val F1: 0.6885 | Val Acc: 0.8136 (New Best)
Epoch 02 | Loss: 0.4196 | Val F1: 0.7805 | Val Acc: 0.9008 (New Best)
Epoch 03 | Loss: 0.2872 | Val F1: 0.8048 | Val Acc: 0.9116 (New Best)
Epoch 04 | Loss: 0.1803 | Val F1: 0.7540 | Val Acc: 0.8825 (Patience: 1/3)
Epoch 05 | Loss: 0.1345 | Val F1: 0.7763 | Val Acc: 0.9090 (Patience: 2/3)
Epoch 06 | Loss: 0.0908 | Val F1: 0.7889 | Val Acc: 0.9109 (Patience: 3/3)
Early stopping triggered after 6 epochs.


# 8. Final Benchmark Evaluation & Classification Reports
Evaluates all trained architectures on the test set and outputs macro F1, overall accuracy, and class-level classification metrics for OCA.

In [9]:
CLASS_NAMES = ["Negative", "Neutral", "Positive"]


def get_predictions(model, loader):
    """Extract ground truth and model predictions on a dataset loader."""
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for batch in loader:
            ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            ont = batch["ontology_features"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(ids, mask, ont)
            _, pr = torch.max(outputs, 1)
            preds.extend(pr.cpu().numpy())
            targets.extend(labels.cpu().numpy())
    return targets, preds


# Collect test predictions
y_true, p1 = get_predictions(m1, test_loader)
_, p2 = get_predictions(m2, test_loader)
_, p3 = get_predictions(m3, test_loader)
_, p4 = get_predictions(m4, test_loader)
_, p5 = get_predictions(m5, test_loader)
_, p_res = get_predictions(m_res, test_loader)
_, p_oca = get_predictions(m_oca, test_loader)

models_list = [
    ("Baseline (ViSoBERT)", "-", p1),
    ("Raw + Concat", "24 (Raw)", p2),
    ("Raw + Gate", "24 (Raw)", p3),
    ("Dense + Concat", "64 (Dense)", p4),
    ("Dense + Gate", "64 (Dense)", p5),
    ("Gated Residual Fusion", "768 (Projected)", p_res),
    ("Ontology Cross-Attention (OCA)", "5 Concepts (768d)", p_oca),
]

summary_records = []
for name, dim, preds in models_list:
    summary_records.append(
        {
            "Model Architecture": name,
            "Ontology Representation": dim,
            "Accuracy": accuracy_score(y_true, preds),
            "Macro F1": f1_score(y_true, preds, average="macro"),
        }
    )

print("\n==================== TEST SET BENCHMARK RESULTS ====================")
df_benchmark = pd.DataFrame(summary_records)
print(df_benchmark.round(4).to_string(index=False))

print("\n========== CLASSIFICATION REPORT (ONTOLOGY CROSS-ATTENTION) ==========")
print(classification_report(y_true, p_oca, target_names=CLASS_NAMES, digits=4))


==================== TEST SET BENCHMARK RESULTS ====================
            Model Architecture Ontology Representation  Accuracy  Macro F1
           Baseline (ViSoBERT)                       -    0.9002    0.7630
                  Raw + Concat                24 (Raw)    0.8834    0.7515
                    Raw + Gate                24 (Raw)    0.8973    0.7640
                Dense + Concat              64 (Dense)    0.9002    0.7587
                  Dense + Gate              64 (Dense)    0.9024    0.7664
         Gated Residual Fusion         768 (Projected)    0.9021    0.7581
Ontology Cross-Attention (OCA)       5 Concepts (768d)    0.8863    0.7564

========== CLASSIFICATION REPORT (ONTOLOGY CROSS-ATTENTION) ==========
              precision    recall  f1-score   support

    Negative     0.8890    0.9439    0.9157      1409
     Neutral     0.4000    0.4910    0.4409       167
    Positive     0.9515    0.8767    0.9126      1590

    accuracy                         0.8

# 9. Neuro-Symbolic Rule-Based Post-Processing
Overrides uncertain neural predictions using high-confidence symbolic signals from the ontology feature representations.

In [10]:
class NeuroSymbolicReasoner:
    """Combines neural probability outputs with ontology polarities for rule-based overrides."""

    def __init__(
        self,
        model,
        engine,
        tokenizer,
        device,
        ontology_threshold=0.55,
        model_conf_limit=0.85,
    ):
        self.model = model
        self.engine = engine
        self.tokenizer = tokenizer
        self.device = device
        self.ont_threshold = ontology_threshold
        self.model_conf_limit = model_conf_limit
        self.idx2label = {0: "Negative", 1: "Neutral", 2: "Positive"}

    def infer(self, text):
        inputs = self.tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=128,
            padding="max_length",
        )
        input_ids = inputs["input_ids"].to(self.device)
        attention_mask = inputs["attention_mask"].to(self.device)

        vec = self.engine.getvector(text)
        if isinstance(vec, tuple):
            vec = vec[0]
        ont_tensor = torch.tensor(np.array([vec]), dtype=torch.float).to(
            self.device
        )

        self.model.eval()
        with torch.no_grad():
            outputs = self.model(input_ids, attention_mask, ont_tensor)
            probs = F.softmax(outputs, dim=1)

        pred_idx = torch.argmax(probs).item()
        confidence = probs[0][pred_idx].item()
        neural_label = self.idx2label[pred_idx]

        # Symbolic override signals: index 19 (Pos), index 20 (Neg)
        pos_signal = vec[19]
        neg_signal = vec[20]
        final_label = neural_label

        # Override low-confidence Neutral predictions when ontology detects strong polarity
        if pred_idx == 1:
            if (
                neg_signal > self.ont_threshold
                and confidence < self.model_conf_limit
            ):
                final_label = "Negative"
            elif (
                pos_signal > self.ont_threshold
                and confidence < self.model_conf_limit
            ):
                final_label = "Positive"

        return neural_label, final_label


reasoner = NeuroSymbolicReasoner(
    model=m1,
    engine=ontengine,
    tokenizer=tokenizer,
    device=device,
    ontology_threshold=0.55,
    model_conf_limit=0.85,
)

y_true_ns = []
y_pred_base = []
y_pred_neuro = []
label_map = {"Negative": 0, "Neutral": 1, "Positive": 2}

for _, row in tqdm(
    df_test.iterrows(), total=len(df_test), desc="Neuro-Symbolic Inference"
):
    text = str(row["Sentence"])
    true_lbl = int(row["Sentiment"])

    base_lbl, neuro_lbl = reasoner.infer(text)
    y_true_ns.append(true_lbl)
    y_pred_base.append(label_map[base_lbl])
    y_pred_neuro.append(label_map[neuro_lbl])

acc_base = accuracy_score(y_true_ns, y_pred_base)
f1_base = f1_score(y_true_ns, y_pred_base, average="macro")
acc_neuro = accuracy_score(y_true_ns, y_pred_neuro)
f1_neuro = f1_score(y_true_ns, y_pred_neuro, average="macro")

df_rule_comparison = pd.DataFrame(
    {
        "Metric": ["Accuracy", "Macro F1"],
        "Baseline (ViSoBERT Only)": [acc_base, f1_base],
        "Neuro-Symbolic (ViSoBERT + Rules)": [acc_neuro, f1_neuro],
        "Delta": [acc_neuro - acc_base, f1_neuro - f1_base],
    }
)

print("\n============ NEURO-SYMBOLIC RULE-BASED POST-PROCESSING ============")
print(df_rule_comparison.round(4).to_string(index=False))

total_overrides = sum(1 for b, n in zip(y_pred_base, y_pred_neuro) if b != n)
print(
    f"\nTotal samples overridden: {total_overrides}/{len(df_test)} ({total_overrides / len(df_test) * 100:.2f}%)"
)

Neuro-Symbolic Inference:   0%|          | 0/3166 [00:00<?, ?it/s]


============ NEURO-SYMBOLIC RULE-BASED POST-PROCESSING ============
  Metric  Baseline (ViSoBERT Only)  Neuro-Symbolic (ViSoBERT + Rules)   Delta
Accuracy                    0.9002                             0.9005  0.0003
Macro F1                    0.7630                             0.7613 -0.0017

Total samples overridden: 21/3166 (0.66%)


# 10. Statistical Significance Testing (Paired Permutation Test)
Calculates exact permutation p-values over 10,000 resamples to evaluate statistical significance ($p < 0.05$) across comparative models against OCA.

In [11]:
from src.significance_test import paired_permutation_test, macro_f1


comparisons = [
    ("m1 (Baseline) vs m_res (Residual)", p1, p_res),
    ("m1 (Baseline) vs m_oca (OCA)", p1, p_oca),
    ("m3 (Raw+Gate) vs m_oca (OCA)", p3, p_oca),
    ("m_res (Residual) vs m_oca (OCA)", p_res, p_oca),
]

print(
    "\n============== STATISTICAL SIGNIFICANCE TESTS (10,000 ITERATIONS) =============="
)

results = []
for name, preds_a, preds_b in comparisons:
    f1_a, f1_b, diff_f1, p_val_f1 = paired_permutation_test(
        y_true, preds_a, preds_b, metric_func=macro_f1
    )
    acc_a, acc_b, diff_acc, p_val_acc = paired_permutation_test(
        y_true, preds_a, preds_b, metric_func=accuracy_score
    )

    sig_f1 = "Significant (p < 0.05)" if p_val_f1 < 0.05 else "Not Significant"
    sig_acc = (
        "Significant (p < 0.05)" if p_val_acc < 0.05 else "Not Significant"
    )

    results.append(
        {
            "Comparison": name,
            "F1 (Model A)": round(f1_a, 4),
            "F1 (Model B)": round(f1_b, 4),
            "Diff (F1)": round(diff_f1, 4),
            "p-val (F1)": round(p_val_f1, 5),
            "Sig (F1)": sig_f1,
            "p-val (Acc)": round(p_val_acc, 5),
            "Sig (Acc)": sig_acc,
        }
    )

df_perm = pd.DataFrame(results)
print(df_perm.to_string(index=False))
print(
    "\n================================================================================"
)


============== STATISTICAL SIGNIFICANCE TESTS (10,000 ITERATIONS) ==============
                       Comparison  F1 (Model A)  F1 (Model B)  Diff (F1)  p-val (F1)        Sig (F1)  p-val (Acc)              Sig (Acc)
m1 (Baseline) vs m_res (Residual)        0.7630        0.7581     0.0049     0.61194 Not Significant      0.67683        Not Significant
     m1 (Baseline) vs m_oca (OCA)        0.7630        0.7564     0.0066     0.52995 Not Significant      0.00270 Significant (p < 0.05)
     m3 (Raw+Gate) vs m_oca (OCA)        0.7640        0.7564     0.0077     0.44556 Not Significant      0.01530 Significant (p < 0.05)
  m_res (Residual) vs m_oca (OCA)        0.7581        0.7564     0.0017     0.87351 Not Significant      0.00040 Significant (p < 0.05)

